# Blocky from Python

Everything the Experiments page does is scriptable. `blocky` is the small client that ships with the box (full REST API: [http://localhost:8000/docs](http://localhost:8000/docs)).

In [ ]:
import blocky
b = blocky.Client()  # this box
b.hardware()

In [ ]:
[(e['id'], e['name'], e['runs']) for e in b.experiments()]

## All results as a table
One row per run, every metric as a column, the same numbers as the Report tab and the CSV export.

In [ ]:
exp = b.experiments()[0]
exp = b.experiment(exp['id'])
df = exp.results()
df[['label', 'speed.tg', 'mmlu.acc', 'kld', 'hw.headroom.vram_model_mb', 'hw.decode.roof_util']]

## Quality vs speed
Each run as a point: further right = faster, higher = smarter.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df['speed.tg'], df['mmlu.acc'] * 100)
for _, r in df.iterrows():
    ax.annotate(r['label'], (r['speed.tg'], r['mmlu.acc'] * 100), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('Decode (tok/s)'); ax.set_ylabel('MMLU-Pro (%)'); ax.grid(alpha=.3)
plt.show()

## Queue new runs from code
Branch from any run (its exact setup + changes), or sweep a grid. They appear on the canvas too.
Uncomment to run; each profile takes ~4–8 minutes.

In [ ]:
# run_id = exp.branch(exp.baseline(), kv='q4_0')
# exp.sweep({'ctx': [4096, 16384, 32768]})
# exp.wait()

## Weights, scans and engines
```python
b.variant('Qwen3.5-9B-Q8_0.gguf', 'Q3_K_M', imatrix=True, overrides={'attn_v': 'q8_0'}).wait()
b.scan('Qwen3.5-0.8B-Q8_0.gguf', mode='quantize', qtype='q2_k').wait()
b.build_engine('https://github.com/ggml-org/llama.cpp', 'master', name='upstream').wait()
```